# Ззадачи 5.2, 5.8

- **5.2.** Составить параметрические уравнения поверхностей вращения: катеноида и псевдосферы.
- **5.8 б, г, и.** Составить уравнение касательной плоскости и нормали к заданным поверхностям в указанных точках.

В списке работ пункт 5.2 продублирован; в каркас включены оба подпункта 5.2 а, б.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sin, cos, tan, sqrt, pi

try:
    import sympy as sp
except ImportError:
    sp = None

EPS = 1e-9
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
def set_axes_equal_3d(ax):
    """Делает масштабы по осям 3D одинаковыми."""
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])
    radius = 0.5 * max([x_range, y_range, z_range])
    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)
    ax.set_xlim3d([x_middle - radius, x_middle + radius])
    ax.set_ylim3d([y_middle - radius, y_middle + radius])
    ax.set_zlim3d([z_middle - radius, z_middle + radius])


def setup_3d(xlim=(-5, 5), ylim=(-5, 5), zlim=(-5, 5), title=None):
    """Создает 3D-систему координат."""
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    if title:
        ax.set_title(title)
    # оси координат
    ax.plot([xlim[0], xlim[1]], [0, 0], [0, 0], linewidth=1)
    ax.plot([0, 0], [ylim[0], ylim[1]], [0, 0], linewidth=1)
    ax.plot([0, 0], [0, 0], [zlim[0], zlim[1]], linewidth=1)
    return fig, ax


def plot_points_3d(ax, points, labels=None):
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], p[2], s=45)
        if label:
            ax.text(p[0], p[1], p[2], '  ' + label)


def plot_parametric_3d(ax, r_func, t_range, n=800, label=None):
    """Рисует пространственную параметрическую кривую t -> (x(t), y(t), z(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    r = np.asarray(r_func(t), dtype=float)
    ax.plot(r[0], r[1], r[2], label=label)
    if label:
        ax.legend()
    return r


def plot_surface_3d(ax, r_func, u_range, v_range, nu=80, nv=80, alpha=0.45, label=None):
    """Рисует параметрическую поверхность r(u,v). r_func должен возвращать массивы X,Y,Z."""
    u = np.linspace(u_range[0], u_range[1], nu)
    v = np.linspace(v_range[0], v_range[1], nv)
    U, V = np.meshgrid(u, v)
    X, Y, Z = r_func(U, V)
    surf = ax.plot_surface(X, Y, Z, alpha=alpha, linewidth=0, antialiased=True)
    if label:
        surf.set_label(label)
    set_axes_equal_3d(ax)
    return X, Y, Z

In [ ]:
def partial_u(r, u, v, h=1e-5):
    return (np.asarray(r(u + h, v)) - np.asarray(r(u - h, v))) / (2*h)


def partial_v(r, u, v, h=1e-5):
    return (np.asarray(r(u, v + h)) - np.asarray(r(u, v - h))) / (2*h)


def second_uu(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v)) - 2*np.asarray(r(u, v)) + np.asarray(r(u - h, v))) / (h*h)


def second_uv(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v + h)) - np.asarray(r(u + h, v - h)) - np.asarray(r(u - h, v + h)) + np.asarray(r(u - h, v - h))) / (4*h*h)


def second_vv(r, u, v, h=1e-4):
    return (np.asarray(r(u, v + h)) - 2*np.asarray(r(u, v)) + np.asarray(r(u, v - h))) / (h*h)


def surface_normal(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    return normalize(np.cross(ru, rv))


def first_fundamental_form(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    E = float(np.dot(ru, ru))
    F = float(np.dot(ru, rv))
    G = float(np.dot(rv, rv))
    return np.array([[E, F], [F, G]])


def second_fundamental_form(r, u, v):
    m = surface_normal(r, u, v)
    L = float(np.dot(second_uu(r, u, v), m))
    M = float(np.dot(second_uv(r, u, v), m))
    N = float(np.dot(second_vv(r, u, v), m))
    return np.array([[L, M], [M, N]])


def tangent_plane_patch(r, u0, v0, su=1.0, sv=1.0, n=12):
    """Патч касательной плоскости через r(u0,v0)."""
    p = np.asarray(r(u0, v0), dtype=float)
    ru = partial_u(r, u0, v0)
    rv = partial_v(r, u0, v0)
    a = np.linspace(-su, su, n)
    b = np.linspace(-sv, sv, n)
    A, B = np.meshgrid(a, b)
    X = p[0] + A*ru[0] + B*rv[0]
    Y = p[1] + A*ru[1] + B*rv[1]
    Z = p[2] + A*ru[2] + B*rv[2]
    return X, Y, Z


def plot_tangent_plane_and_normal(ax, r, u0, v0, plane_scale=0.5, normal_scale=1.0):
    p = np.asarray(r(u0, v0), dtype=float)
    X, Y, Z = tangent_plane_patch(r, u0, v0, plane_scale, plane_scale)
    ax.plot_surface(X, Y, Z, alpha=0.30, linewidth=0)
    m = surface_normal(r, u0, v0)
    ax.quiver(p[0], p[1], p[2], normal_scale*m[0], normal_scale*m[1], normal_scale*m[2], arrow_length_ratio=0.15)
    plot_points_3d(ax, [p], ['M'])
    set_axes_equal_3d(ax)
    return p, m

## Задача 5.2а — катеноид

In [ ]:
a = 1.0

def catenoid(u, v, a=a):
    return (a*np.cosh(u/a)*np.cos(v), a*np.cosh(u/a)*np.sin(v), u)

fig, ax = setup_3d((-4, 4), (-4, 4), (-3, 3), title='5.2а Катеноид')
plot_surface_3d(ax, catenoid, (-2.2, 2.2), (0, 2*np.pi), alpha=0.55)
plt.show()

## Задача 5.2б — псевдосфера

In [ ]:
a = 1.0

def pseudosphere(u, v, a=a):
    # u берется из (0, pi); рядом с 0 и pi есть особенности.
    return (a*np.sin(u)*np.cos(v),
            a*np.sin(u)*np.sin(v),
            a*(np.log(np.tan(u/2)) + np.cos(u)))

fig, ax = setup_3d((-1.3, 1.3), (-1.3, 1.3), (-4, 4), title='5.2б Псевдосфера')
plot_surface_3d(ax, pseudosphere, (0.15, 2.75), (0, 2*np.pi), alpha=0.55)
plt.show()

## Задача 5.8б — параметрическая поверхность

In [ ]:
def surface_58b(u, v):
    return np.array([u + v, u**2 - 2*v, u**3 - u*v], dtype=float)

u0, v0 = 1.0, 2.0
M = surface_58b(u0, v0)
print('M =', M)
print('I =\n', first_fundamental_form(surface_58b, u0, v0))
print('normal =', surface_normal(surface_58b, u0, v0))

fig, ax = setup_3d((-1, 5), (-6, 3), (-5, 3), title='5.8б Касательная плоскость и нормаль')
plot_surface_3d(ax, surface_58b, (-0.5, 2.2), (0.0, 3.2), alpha=0.35)
plot_tangent_plane_and_normal(ax, surface_58b, u0, v0, plane_scale=0.35, normal_scale=1.2)
plt.show()

## Задача 5.8г — поверхность $r(u,v)=(au,\sin u,bv)$

In [ ]:
a, b = 1.2, 0.8
u0, v0 = 1.0, 0.5  # произвольная точка

def surface_58g(u, v, a=a, b=b):
    return np.array([a*u, np.sin(u), b*v], dtype=float)

print('M =', surface_58g(u0, v0))
print('normal =', surface_normal(surface_58g, u0, v0))

fig, ax = setup_3d((-3, 3), (-2, 2), (-3, 3), title='5.8г Касательная плоскость и нормаль')
plot_surface_3d(ax, surface_58g, (-2.0, 2.0), (-2.0, 2.0), alpha=0.35)
plot_tangent_plane_and_normal(ax, surface_58g, u0, v0, plane_scale=0.45, normal_scale=1.0)
plt.show()

## Задача 5.8и — эллиптический параболоид $x^2/a^2+y^2/b^2=2z$

In [ ]:
a, b = 2.0, 1.0
x0, y0 = 1.0, 0.6
z0 = 0.5*(x0**2/a**2 + y0**2/b**2)
M = np.array([x0, y0, z0])

# Явная параметризация графика z=(x^2/a^2+y^2/b^2)/2.
def paraboloid(x, y, a=a, b=b):
    return np.array([x, y, 0.5*(x**2/a**2 + y**2/b**2)], dtype=float)

# Для неявной поверхности F=x^2/a^2+y^2/b^2-2z нормаль равна grad F.
def grad_F_58i(x, y, z, a=a, b=b):
    return np.array([2*x/a**2, 2*y/b**2, -2.0], dtype=float)

normal = normalize(grad_F_58i(x0, y0, z0))
print('M =', M)
print('normal =', normal)

fig, ax = setup_3d((-3, 3), (-3, 3), (-0.2, 2.5), title='5.8и Касательная плоскость и нормаль')
plot_surface_3d(ax, paraboloid, (-2.5, 2.5), (-2.5, 2.5), alpha=0.35)
plot_tangent_plane_and_normal(ax, paraboloid, x0, y0, plane_scale=0.5, normal_scale=0.8)
plt.show()

## Место для ручной записи уравнений

Для параметрической поверхности:

$$R = r(u_0,v_0)+\alpha r_u(u_0,v_0)+\beta r_v(u_0,v_0),\qquad n=r_u\times r_v.$$

Касательная плоскость:

$$(R-r_0,n)=0.$$

Нормаль:

$$R=r_0+tn.$$